In [19]:
import json
import pandas as pd
from pathlib import Path

In [20]:
baseline_top50 = pd.read_parquet("../../data/processed/retrieval/embedding_baseline_clean_top50.parquet")

two_tower_top50 = pd.read_parquet(
    "../artifacts/models/two_tower_text_v1/evaluation/top50_retrieval.parquet"
)

baseline_match = pd.read_parquet("../../data/processed/retrieval/embedding_baseline_clean_top50.parquet")
two_tower_match = pd.read_parquet(
    "../artifacts/models/two_tower_text_v1/evaluation/match_metrics.parquet"
)
two_tower_score = pd.read_parquet(
    "../artifacts/models/two_tower_text_v1/evaluation/score_metrics.parquet"
)

queries = pd.read_json("../../data/processed/retrieval/query_intents.jsonl", lines=True)

print("Baseline top50 rows:", len(baseline_top50))
print("Two-tower top50 rows:", len(two_tower_top50))
print("Queries:", len(queries))

Baseline top50 rows: 1050
Two-tower top50 rows: 1050
Queries: 21


In [21]:
def compute_metrics_df(retrieval_df: pd.DataFrame, queries_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    match_rows = []
    score_rows = []

    for _, q in queries_df.iterrows():
        qid = q["query_id"]
        strategy = q["strategy"]
        score_col = f"{strategy}_score"

        qdf = retrieval_df[retrieval_df["query_id"] == qid].copy()
        top10 = qdf.head(10)
        top20 = qdf.head(20)

        match_rows.append(
            {
                "query_id": qid,
                "strategy": strategy,
                "top10_match_rate": (top10["top_strategy"] == strategy).mean(),
                "top20_match_rate": (top20["top_strategy"] == strategy).mean(),
            }
        )

        score_rows.append(
            {
                "query_id": qid,
                "strategy": strategy,
                "top20_mean_score": top20[score_col].mean(),
                "top20_median_score": top20[score_col].median(),
                "top20_high_score_rate": (top20[score_col] >= 70).mean(),
            }
        )

    return pd.DataFrame(match_rows), pd.DataFrame(score_rows)

In [22]:
baseline_match_df, baseline_score_df = compute_metrics_df(baseline_top50, queries)
baseline_match_df

,query_id,strategy,top10_match_rate,top20_match_rate
0,q001,single_dwelling_rebuild,0.6,0.30
1,q002,single_dwelling_rebuild,0.1,0.05
2,q003,single_dwelling_rebuild,0.7,0.85
3,q004,assembly_opportunity,0.0,0.00
4,q005,assembly_opportunity,0.0,0.00
5,q006,assembly_opportunity,0.0,0.00
6,q007,granny_flat,0.0,0.00
7,q008,granny_flat,0.0,0.00
8,q009,granny_flat,0.0,0.00
9,q010,land_bank_hold,0.1,0.10


In [23]:
baseline_summary = {
    "top10_match_rate_mean": baseline_match_df["top10_match_rate"].mean(),
    "top20_match_rate_mean": baseline_match_df["top20_match_rate"].mean(),
    "top20_mean_score_mean": baseline_score_df["top20_mean_score"].mean(),
    "top20_median_score_mean": baseline_score_df["top20_median_score"].mean(),
    "top20_high_score_rate_mean": baseline_score_df["top20_high_score_rate"].mean(),
}

two_tower_summary = {
    "top10_match_rate_mean": two_tower_match["top10_match_rate"].mean(),
    "top20_match_rate_mean": two_tower_match["top20_match_rate"].mean(),
    "top20_mean_score_mean": two_tower_score["top20_mean_score"].mean(),
    "top20_median_score_mean": two_tower_score["top20_median_score"].mean(),
    "top20_high_score_rate_mean": two_tower_score["top20_high_score_rate"].mean(),
}

pd.DataFrame([baseline_summary, two_tower_summary], index=["baseline_clean", "two_tower"])

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean
baseline_clean,0.176190,0.161905,52.603333,53.895238,0.228571
two_tower,0.219048,0.192857,70.569048,70.976190,0.519048


In [24]:
compare_match = baseline_match_df.merge(
    two_tower_match,
    on=["query_id", "strategy"],
    suffixes=("_baseline", "_two_tower"),
)

compare_match["top10_gain"] = (
    compare_match["top10_match_rate_two_tower"] - compare_match["top10_match_rate_baseline"]
)
compare_match["top20_gain"] = (
    compare_match["top20_match_rate_two_tower"] - compare_match["top20_match_rate_baseline"]
)

compare_match.sort_values("top10_gain", ascending=False)

,query_id,strategy,top10_match_rate_baseline,top20_match_rate_baseline,top10_match_rate_two_tower,top20_match_rate_two_tower,top10_gain,top20_gain
17,q018,low_rise_apartment,0.0,0.00,1.0,1.00,1.0,1.00
16,q017,low_rise_apartment,0.0,0.00,1.0,1.00,1.0,1.00
15,q016,low_rise_apartment,0.0,0.00,1.0,1.00,1.0,1.00
11,q012,land_bank_hold,0.6,0.60,1.0,0.75,0.4,0.15
20,q021,dual_occupancy,0.0,0.00,0.0,0.00,0.0,0.00
14,q015,townhouse_multi_dwelling,0.0,0.00,0.0,0.00,0.0,0.00
3,q004,assembly_opportunity,0.0,0.00,0.0,0.00,0.0,0.00
4,q005,assembly_opportunity,0.0,0.00,0.0,0.00,0.0,0.00
5,q006,assembly_opportunity,0.0,0.00,0.0,0.00,0.0,0.00
6,q007,granny_flat,0.0,0.00,0.0,0.00,0.0,0.00


In [25]:
compare_score = baseline_score_df.merge(
    two_tower_score,
    on=["query_id", "strategy"],
    suffixes=("_baseline", "_two_tower"),
)

compare_score["top20_mean_score_gain"] = (
    compare_score["top20_mean_score_two_tower"] - compare_score["top20_mean_score_baseline"]
)
compare_score["top20_high_score_rate_gain"] = (
    compare_score["top20_high_score_rate_two_tower"] - compare_score["top20_high_score_rate_baseline"]
)

compare_score.sort_values("top20_mean_score_gain", ascending=False)

,query_id,strategy,top20_mean_score_baseline,top20_median_score_baseline,top20_high_score_rate_baseline,top20_mean_score_two_tower,top20_median_score_two_tower,top20_high_score_rate_two_tower,top20_mean_score_gain,top20_high_score_rate_gain
10,q011,land_bank_hold,31.650,34.0,0.00,85.800,93.0,0.70,54.150,0.70
13,q014,townhouse_multi_dwelling,38.550,39.5,0.00,90.000,90.0,1.00,51.450,1.00
3,q004,assembly_opportunity,44.330,43.0,0.05,89.400,93.0,0.90,45.070,0.85
5,q006,assembly_opportunity,46.750,45.5,0.00,89.400,93.0,0.90,42.650,0.90
4,q005,assembly_opportunity,47.650,41.5,0.25,89.400,93.0,0.90,41.750,0.65
9,q010,land_bank_hold,53.850,53.6,0.00,93.000,93.0,1.00,39.150,1.00
12,q013,townhouse_multi_dwelling,52.550,53.0,0.00,90.000,90.0,1.00,37.450,1.00
17,q018,low_rise_apartment,60.550,72.5,0.80,97.500,97.5,1.00,36.950,0.20
16,q017,low_rise_apartment,72.500,72.5,1.00,97.500,97.5,1.00,25.000,0.00
15,q016,low_rise_apartment,75.875,77.5,0.95,97.500,97.5,1.00,21.625,0.05


In [26]:
strategy_gain = compare_match.groupby("strategy", as_index=False).agg(
    top10_gain_mean=("top10_gain", "mean"),
    top20_gain_mean=("top20_gain", "mean"),
)

strategy_score_gain = compare_score.groupby("strategy", as_index=False).agg(
    top20_mean_score_gain_mean=("top20_mean_score_gain", "mean"),
    top20_high_score_rate_gain_mean=("top20_high_score_rate_gain", "mean"),
)

strategy_gain.merge(strategy_score_gain, on="strategy").sort_values(
    "top20_mean_score_gain_mean", ascending=False
)

,strategy,top10_gain_mean,top20_gain_mean,top20_mean_score_gain_mean,top20_high_score_rate_gain_mean
0,assembly_opportunity,0.000000,0.000000,43.156667,0.800000
3,land_bank_hold,0.066667,-0.116667,36.443333,0.583333
6,townhouse_multi_dwelling,0.000000,0.000000,30.516667,0.616667
4,low_rise_apartment,1.000000,1.000000,27.858333,0.083333
2,granny_flat,0.000000,0.000000,8.288333,0.100000
1,dual_occupancy,-0.300000,-0.266667,-7.323333,0.000000
5,single_dwelling_rebuild,-0.466667,-0.400000,-13.180000,-0.150000


In [27]:
view_cols = [
    "address",
    "primary_zoning_code",
    "lot_size_band",
    "constraint_severity_band",
    "station_distance_band",
    "top_strategy",
    "top_strategy_score",
    "similarity",
]

def compare_query(query_id: str, n: int = 10):
    q = queries[queries["query_id"] == query_id].iloc[0]
    print("QUERY:", q["query_id"], "|", q["strategy"])
    print(q["text"])

    print("\nBASELINE")
    display(
        baseline_top50[baseline_top50["query_id"] == query_id][view_cols].head(n)
    )

    print("\nTWO-TOWER")
    display(
        two_tower_top50[two_tower_top50["query_id"] == query_id][view_cols].head(n)
    )

In [28]:
compare_match.sort_values("top10_gain", ascending=False).head(10)

,query_id,strategy,top10_match_rate_baseline,top20_match_rate_baseline,top10_match_rate_two_tower,top20_match_rate_two_tower,top10_gain,top20_gain
17,q018,low_rise_apartment,0.0,0.0,1.0,1.00,1.0,1.00
16,q017,low_rise_apartment,0.0,0.0,1.0,1.00,1.0,1.00
15,q016,low_rise_apartment,0.0,0.0,1.0,1.00,1.0,1.00
11,q012,land_bank_hold,0.6,0.6,1.0,0.75,0.4,0.15
20,q021,dual_occupancy,0.0,0.0,0.0,0.00,0.0,0.00
14,q015,townhouse_multi_dwelling,0.0,0.0,0.0,0.00,0.0,0.00
3,q004,assembly_opportunity,0.0,0.0,0.0,0.00,0.0,0.00
4,q005,assembly_opportunity,0.0,0.0,0.0,0.00,0.0,0.00
5,q006,assembly_opportunity,0.0,0.0,0.0,0.00,0.0,0.00
6,q007,granny_flat,0.0,0.0,0.0,0.00,0.0,0.00


In [29]:
compare_query("q004")

QUERY: q004 | assembly_opportunity
strategy assembly_opportunity | strategic land aggregation potential | future redevelopment upside | larger site preferred | mixed context acceptable | access helpful | not necessarily immediate build

BASELINE


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
150,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.661357
151,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.661357
152,10/79-83 FAUNCE STREET WEST GOSFORD,R1,l,high,within_800m,dual_occupancy,66.0,0.660901
153,1 FEDERATION BOULEVARD WARNERVALE,R1,m,high,within_800m,single_dwelling_rebuild,60.0,0.660039
154,602/72 DONNISON STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,61.0,0.660039
155,22 JASMYNE STREET LISMORE,R1,m,high,within_800m,dual_occupancy,61.0,0.660039
156,13/77 FAUNCE STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,66.0,0.658973
157,5/85-87 FAUNCE STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,66.0,0.658973
158,5/6 MARGIN STREET GOSFORD,R1,m,high,within_800m,single_dwelling_rebuild,65.0,0.658973
159,42 KARALTA ROAD ERINA,R1,m,high,within_800m,single_dwelling_rebuild,65.0,0.658973



TWO-TOWER


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
150,24/80 BONAR STREET WOLLI CREEK,R4,xl,high,within_800m,land_bank_hold,69.0,0.727545
151,113/88 BONAR STREET WOLLI CREEK,R4,xl,high,within_800m,land_bank_hold,69.0,0.727545
152,612/5 BIDJIGAL ROAD ARNCLIFFE,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
153,29/6-8 CULWORTH AVENUE KILLARA,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
154,85/9-19 AMOR STREET ASQUITH,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
155,137/18-34 WAVERLEY STREET BONDI JUNCTION,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
156,60/95 BONAR STREET WOLLI CREEK,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
157,50 BELMONT STREET SUTHERLAND,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
158,11/381-389 KINGSWAY CARINGBAH,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055
159,506/43 SHORELINE DRIVE RHODES,R4,xl,low,within_800m,low_rise_apartment,97.5,0.720055


In [30]:
compare_query("q013")

QUERY: q013 | townhouse_multi_dwelling
strategy townhouse_multi_dwelling | medium density redevelopment | prefers R3 MU1 and some R1 context | larger lot preferred | access helpful | not apartment scale

BASELINE


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
600,126 CRIMEA ROAD MARSFIELD,R4,l,high,800m_2km,land_bank_hold,68.6,0.733014
601,19/36-38 BUSACO ROAD MARSFIELD,R4,l,high,2km_5km,land_bank_hold,57.6,0.732303
602,20/38 COPE STREET LANE COVE,R4,l,moderate,2km_5km,land_bank_hold,74.6,0.731120
603,3/9-11 GARDEN STREET TELOPEA,R4,l,moderate,2km_5km,land_bank_hold,74.6,0.731120
604,12/2-4 TELOPEA STREET TELOPEA,R4,l,moderate,2km_5km,land_bank_hold,74.6,0.731120
605,10/5 GARDEN STREET TELOPEA,R4,l,moderate,2km_5km,land_bank_hold,74.6,0.731120
606,17/40 COPE STREET LANE COVE,R4,l,moderate,2km_5km,land_bank_hold,74.6,0.731120
607,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.730489
608,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.730489
609,17 BLAXLAND STREET LITHGOW,R1,l,high,800m_2km,single_dwelling_rebuild,57.2,0.729786



TWO-TOWER


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
600,1217/671 GARDENERS ROAD MASCOT,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
601,1007/9 GAY STREET CASTLE HILL,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
602,520/2-4 LACHLAN STREET WATERLOO,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
603,47/26-30 MARIAN STREET KILLARA,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
604,134/213 PRINCES HIGHWAY ARNCLIFFE,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
605,22 THIRD AVENUE BLACKTOWN,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
606,1012/12 EAST STREET GRANVILLE,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
607,11/21 RYAN AVENUE SINGLETON,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
608,19/21 RYAN AVENUE SINGLETON,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392
609,505/675 GARDENERS ROAD MASCOT,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.737392


In [31]:
compare_query("q019")

QUERY: q019 | dual_occupancy
strategy dual_occupancy | zoning preference low_dev or mid_dev | lot size preference m l xl | moderate residential intensification | avoid heritage flood severe bushfire

BASELINE


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
900,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.716985
901,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.716985
902,3/51-53 BURNS BAY ROAD LANE COVE,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
903,2/2A HAIG AVENUE GEORGES HALL,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
904,52 SIMMAT AVENUE CONDELL PARK,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
905,10/38-44 SHERWOOD ROAD MERRYLANDS WEST,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
906,16/8-16 EIGHTH AVENUE CAMPSIE,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
907,24/121-127 CANTERBURY ROAD CANTERBURY,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
908,905/10B CHARLES STREET CANTERBURY,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583
909,506/10B CHARLES STREET CANTERBURY,E1,l,low,2km_5km,land_bank_hold,78.6,0.716583



TWO-TOWER


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
900,602/91C BRIDGE ROAD WESTMEAD,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
901,33/91A-97 LONGFIELD STREET CABRAMATTA,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
902,144/97 BONAR STREET WOLLI CREEK,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
903,38/81-85 FLORA STREET KIRRAWEE,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
904,406/19 EPPING ROAD EPPING,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
905,73/97 BONAR STREET WOLLI CREEK,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
906,604/36 VICTORIA STREET EPPING,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
907,504/33 THE GRAND PARADE SUTHERLAND,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
908,33/10-18 HUME STREET WOLLSTONECRAFT,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
909,47/7 HERBERT STREET ST LEONARDS,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021


In [32]:
bad_queries = compare_match.sort_values("top10_match_rate_two_tower", ascending=True)
bad_queries.head(10)

,query_id,strategy,top10_match_rate_baseline,top20_match_rate_baseline,top10_match_rate_two_tower,top20_match_rate_two_tower,top10_gain,top20_gain
0,q001,single_dwelling_rebuild,0.6,0.3,0.0,0.0,-0.6,-0.3
18,q019,dual_occupancy,0.2,0.1,0.0,0.0,-0.2,-0.1
14,q015,townhouse_multi_dwelling,0.0,0.0,0.0,0.0,0.0,0.0
13,q014,townhouse_multi_dwelling,0.0,0.0,0.0,0.0,0.0,0.0
12,q013,townhouse_multi_dwelling,0.0,0.0,0.0,0.0,0.0,0.0
19,q020,dual_occupancy,0.7,0.7,0.0,0.0,-0.7,-0.7
9,q010,land_bank_hold,0.1,0.1,0.0,0.0,-0.1,-0.1
8,q009,granny_flat,0.0,0.0,0.0,0.0,0.0,0.0
20,q021,dual_occupancy,0.0,0.0,0.0,0.0,0.0,0.0
6,q007,granny_flat,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
for qid in bad_queries.head(3)["query_id"].tolist():
    compare_query(qid)

QUERY: q001 | single_dwelling_rebuild
strategy single_dwelling_rebuild | zoning preference low_dev or mid_dev | lot size preference s m l | access preference not critical | avoid heritage flood severe bushfire

BASELINE


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
0,1A GOLDSMITH STREET WOONGARRAH,R1,s,high,2km_5km,single_dwelling_rebuild,45.2,0.684092
1,19/36-38 BUSACO ROAD MARSFIELD,R4,l,high,2km_5km,land_bank_hold,57.6,0.683765
2,2/34 SCOTT STREET HARRINGTON,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753
3,41 MACQUARIE DRIVE BURRILL LAKE,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753
4,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.683339
5,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.683339
6,10 ADELAIDE CLOSE WINGHAM,R1,l,high,over_10km,single_dwelling_rebuild,54.0,0.683283
7,13 ALLUMBA CLOSE TAREE,R1,l,high,over_10km,single_dwelling_rebuild,54.0,0.683283
8,602/72 DONNISON STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,61.0,0.682759
9,1 FEDERATION BOULEVARD WARNERVALE,R1,m,high,within_800m,single_dwelling_rebuild,60.0,0.682759



TWO-TOWER


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
0,2 JARVIS CIRCUIT MACQUARIE PARK,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
1,705/16 EAST STREET GRANVILLE,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
2,229/811 ELIZABETH STREET ZETLAND,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
3,801/9 GAY STREET CASTLE HILL,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
4,310/116 JOYNTON AVENUE ZETLAND,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
5,152/8-12 THOMAS STREET WAITARA,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
6,240/213 PRINCES HIGHWAY ARNCLIFFE,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
7,33/1A PREMIER LANE ROOTY HILL,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
8,2/20 PENNANT STREET CASTLE HILL,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769
9,5/9-13 SURREY STREET MINTO,MU1,xl,low,within_800m,low_rise_apartment,97.5,0.710769


QUERY: q019 | dual_occupancy
strategy dual_occupancy | zoning preference low_dev or mid_dev | lot size preference m l xl | moderate residential intensification | avoid heritage flood severe bushfire

BASELINE


...

,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
900,602/91C BRIDGE ROAD WESTMEAD,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
901,33/91A-97 LONGFIELD STREET CABRAMATTA,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
902,144/97 BONAR STREET WOLLI CREEK,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
903,38/81-85 FLORA STREET KIRRAWEE,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
904,406/19 EPPING ROAD EPPING,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
905,73/97 BONAR STREET WOLLI CREEK,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
906,604/36 VICTORIA STREET EPPING,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
907,504/33 THE GRAND PARADE SUTHERLAND,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
908,33/10-18 HUME STREET WOLLSTONECRAFT,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021
909,47/7 HERBERT STREET ST LEONARDS,R4,xl,low,within_800m,low_rise_apartment,97.5,0.723021


QUERY: q015 | townhouse_multi_dwelling
strategy townhouse_multi_dwelling | multi dwelling but not high rise or apartment intensity | medium density zoning preferred | larger site | station access helpful but not essential

BASELINE


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
700,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.446841
701,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.446841
702,40/554-558 PACIFIC HIGHWAY MOUNT COLAH,R4,l,moderate,within_800m,land_bank_hold,89.0,0.446345
703,25/554-558 PACIFIC HIGHWAY MOUNT COLAH,R4,l,moderate,within_800m,land_bank_hold,89.0,0.446345
704,27/39-41 RAILWAY PARADE ENGADINE,R4,l,moderate,within_800m,land_bank_hold,89.0,0.446345
705,28/39-41 RAILWAY PARADE ENGADINE,R4,l,moderate,within_800m,land_bank_hold,89.0,0.446345
706,10/79-83 FAUNCE STREET WEST GOSFORD,R1,l,high,within_800m,dual_occupancy,66.0,0.444597
707,26/19-23 GRAHAM ROAD NARWEE,R4,l,low,within_800m,land_bank_hold,88.0,0.443814
708,16/13-17 HAMPDEN STREET BEVERLY HILLS,R4,l,low,within_800m,land_bank_hold,88.0,0.443814
709,9/43-45 NEIL STREET MERRYLANDS,R4,l,low,within_800m,land_bank_hold,88.0,0.443814



TWO-TOWER


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
700,113/88 BONAR STREET WOLLI CREEK,R4,xl,high,within_800m,land_bank_hold,69.0,0.702543
701,24/80 BONAR STREET WOLLI CREEK,R4,xl,high,within_800m,land_bank_hold,69.0,0.702543
702,1201/97 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
703,48/95-97 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
704,252/95 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
705,230/95 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
706,609/97 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
707,2 BLAXLAND ROAD RHODES,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
708,910/97 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155
709,906/97 GRAFTON STREET BONDI JUNCTION,MU1,xl,high,within_800m,land_bank_hold,69.0,0.696155


In [34]:
def top_strategy_distribution(retrieval_df: pd.DataFrame, query_id: str, n: int = 20):
    qdf = retrieval_df[retrieval_df["query_id"] == query_id].head(n)
    return qdf["top_strategy"].value_counts(dropna=False)

top_strategy_distribution(two_tower_top50, "q013", 20)

top_strategy
low_rise_apartment    20
Name: count, dtype: int64

In [35]:
out_dir = Path("../../artifacts/models/two_tower_text_v1/evaluation")
out_dir.mkdir(parents=True, exist_ok=True)

compare_match.to_parquet(out_dir / "baseline_vs_two_tower_match_compare.parquet", index=False)
compare_score.to_parquet(out_dir / "baseline_vs_two_tower_score_compare.parquet", index=False)

In [36]:
print(pd.DataFrame([baseline_summary, two_tower_summary], index=["baseline_clean", "two_tower"]))
print(strategy_gain.merge(strategy_score_gain, on="strategy").sort_values(
    "top20_mean_score_gain_mean", ascending=False
))

                top10_match_rate_mean  top20_match_rate_mean  \
baseline_clean               0.176190               0.161905   
two_tower                    0.219048               0.192857   

                top20_mean_score_mean  top20_median_score_mean  \
baseline_clean              52.603333                53.895238   
two_tower                   70.569048                70.976190   

                top20_high_score_rate_mean  
baseline_clean                    0.228571  
two_tower                         0.519048  
                   strategy  top10_gain_mean  top20_gain_mean  \
0      assembly_opportunity         0.000000         0.000000   
3            land_bank_hold         0.066667        -0.116667   
6  townhouse_multi_dwelling         0.000000         0.000000   
4        low_rise_apartment         1.000000         1.000000   
2               granny_flat         0.000000         0.000000   
1            dual_occupancy        -0.300000        -0.266667   
5   single_dwellin